(mmm_roas)=
# Mitigating Unobserved Confounders in MMMs with Lift Test Likelihoods

## Motivation: Why Lift Test Calibration Matters for MMMs

Media Mix Models (MMMs) are valuable when they can answer causal questions such as "What happens to sales if we cut channel X's budget by 50%?" Answering these questions correctly requires the model to capture the true causal structure behind the data. In practice, however, *unobserved confounders*, variables that influence both ad spend and sales but are absent from the model, are nearly unavoidable. Because they are not accounted for, they bias the estimated return on ad spend (ROAS) and can lead to poor budget decisions.

Classical remedies like instrumental variables are rarely feasible in marketing settings because valid instruments are hard to find. A more practical alternative is to calibrate the model with *lift tests*: controlled experiments that measure the incremental effect of a marketing intervention. Bayesian inference makes this calibration natural as lift-test results can be folded in as informative priors or likelihood terms that pull the estimates toward experimentally grounded values.

### Background: ROAS Reparametrization (Zhang et al. 2024)

[Zhang et al. (2024)](https://research.google/pubs/media-mix-model-calibration-with-bayesian-priors/) propose reparametrizing the MMM's channel coefficients in terms of ROAS and placing Bayesian priors on those ROAS parameters informed by lift-test outcomes. The blog post [Media Mix Model and Experimental Calibration: A Simulation Study](https://juanitorduz.github.io/mmm_roas/) provides a PyMC implementation of this method and shows that it can partially mitigate confounding bias. However, the approach has a limitation: it does not automatically improve as more lift tests become available. The authors suggest aggregating multiple experiments via a weighted average of ROAS estimates, which does not fully exploit each test's information.

### Our Approach: Calibration via Saturation Curve Likelihoods

In this notebook we showcase a different route available in PyMC-Marketing. Instead of reparametrizing in terms of ROAS, we add *custom likelihood terms* on the saturation curves of each channel using the observed lift-test results. Each new lift test contributes an additional likelihood term, so the model's estimates improve incrementally as experiments accumulate, no manual aggregation required. This approach is explain in detail in {ref}`mmm_lift_test`.

Concretely, we will:

- Fit a **baseline (uncalibrated) model** and show how the unobserved confounder biases ROAS estimates.
- Fit a **calibrated model** that incorporates lift tests as saturation-curve likelihoods and demonstrate that ROAS estimates converge toward their true values.
- Evaluate both models with **time-slice cross-validation** to confirm that as lift tests become available, the ROAS estimates stabilize and converge to the true ROAS.


## Prepare Notebook

In [ ]:
import warnings

import arviz as az
import arviz_plots as azp
import graphviz as gr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr
from matplotlib.lines import Line2D
from pymc_extras.prior import Prior
from xarray import DataArray

from pymc_marketing.hsgp_kwargs import HSGPKwargs
from pymc_marketing.metrics import crps
from pymc_marketing.mmm import MMM, GeometricAdstock, LogisticSaturation
from pymc_marketing.mmm.time_slice_cross_validation import TimeSliceCrossValidator
from pymc_marketing.paths import data_dir

warnings.filterwarnings("ignore", category=FutureWarning)


az.style.use("arviz-darkgrid")
plt.rcParams["figure.figsize"] = [12, 7]
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.facecolor"] = "white"


%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [ ]:
seed: int = sum(map(ord, "mmm_roas_notebook"))
rng: np.random.Generator = np.random.default_rng(seed=seed)

## Read Data

We read the data, which is available in the `data` directory of our repository.

In [ ]:
data_path = data_dir / "mmm_roas_data.csv"

raw_df = pd.read_csv(data_path, parse_dates=["date"])

raw_df.info()

There are many columns in the dataset. We will explain them below. For now, what is important for modeling purposes is that we will only use the `date`, `x1`, `x2` (channels) and `y` (target) columns.

In [ ]:
model_df = raw_df.copy().filter(["date", "x1", "x2", "y"])

## Data Generating Process  

In the original [blog post](https://juanitorduz.github.io/mmm_roas/), the authors generate the data using the following DAG.

In [ ]:
g = gr.Digraph()
g.node(name="seasonality", label="seasonality", color="lightgray", style="filled")
g.node(name="trend", label="trend")
g.node(name="z", label="z", color="lightgray", style="filled")
g.node(name="x1", label="x1", color="#2a2eec80", style="filled")
g.node(name="x2", label="x2", color="#fa7c1780", style="filled")
g.node(name="y", label="y", color="#328c0680", style="filled")
g.edge(tail_name="seasonality", head_name="x1")
g.edge(tail_name="z", head_name="x1")
g.edge(tail_name="x1", head_name="y")
g.edge(tail_name="seasonality", head_name="y")
g.edge(tail_name="trend", head_name="y")
g.edge(tail_name="z", head_name="y")
g.edge(tail_name="x2", head_name="y")
g

We are interested in the effect of the `x1` and `x2` channels on the `y` variable. We have additional covariates like yearly seasonality and a non-linear trend component. In addition, there is an unobserved confounder `z` that affects both the channels and the target variable. This variable introduces a bias in the estimates if we do not account for it. However, for this problem we are going to assume that we do not have access to the `z` variable (hence, unobserved).

In the `raw_df` we have all the columns needed in the data generating process to obtain the target variable `y`. For example, the `x1_adstock_saturated` column is the result of applying the adstock function and then the saturation to the `x1` channel.

The target variable `y` is generated as  

```
y = amplitude * (trend + seasonality + z + x1_effect + x2_effect + epsilon)
```

where `epsilon` is a Gaussian noise and `amplitude` is set to $100$.

The variables `y01` and `y02` are the target variable `y` without the effect of the `x1` and `x2` channels, respectively. Hence

```
y01 = amplitude * (trend + seasonality + z + x2_effect + epsilon)
y02 = amplitude * (trend + seasonality + z + x1_effect + epsilon)
```

 For details on the data generating process, please refer to the [blog post](https://juanitorduz.github.io/mmm_roas/).

From the variables `y01` and `y02` we can compute the true ROAS for the `x1` and `x2` channels for the whole period (for simplicity we ignore the carryover effect).

In [ ]:
true_roas_x1 = (raw_df["y"] - raw_df["y01"]).sum() / raw_df["x1"].sum()
true_roas_x2 = (raw_df["y"] - raw_df["y02"]).sum() / raw_df["x2"].sum()

print(f"True ROAS for x1: {true_roas_x1:.2f}")
print(f"True ROAS for x2: {true_roas_x2:.2f}")

We would like to recover the true ROAS for the `x1` and `x2` channels using a media mix model.

Before jumping into the model, let's plot the target data and the channels data to understand it better.

In [ ]:
fig, ax = plt.subplots(
    nrows=2,
    ncols=1,
    sharex=True,
    sharey=False,
    layout="constrained",
)

sns.lineplot(
    x="date",
    y="y",
    data=model_df,
    color="black",
    ax=ax[0],
)
ax[0].set_title("Target Data")

model_df.melt(
    id_vars=["date"], value_vars=["x1", "x2"], var_name="channel", value_name="x"
).pipe(
    (sns.lineplot, "data"),
    x="date",
    y="x",
    hue="channel",
    ax=ax[1],
)
ax[1].legend(title="Channel", title_fontsize=12)
ax[1].set_title("Channels Spend Data");

## Baseline Model

To begin with, we fit a media mix model, without including the unobserved confounder `z`, using a geometric adstock and logistic saturation using PyMC-Marketing's API. For more details on the model, please refer to the {ref}`mmm_example` notebook. As we discuss in that notebook, without any lift test information, a good starting point is to pass the cost share into the prior of the beta channel coefficients.

In [ ]:
cost_share = DataArray(
    model_df[["x1", "x2"]].sum() / model_df[["x1", "x2"]].sum().sum(),
    dims="channel",
)

We can specify the model configuration as follows (see the {ref}`model_configuration` notebook for more details): 

In [ ]:
model_config = {
    "likelihood": Prior("Normal", sigma=Prior("HalfNormal", sigma=2)),
    "gamma_fourier": Prior("Normal", mu=0, sigma=2, dims="fourier_mode"),
    "intercept_tvp_config": HSGPKwargs(
        m=50, L=None, eta_lam=1.0, ls_mu=5.0, ls_sigma=10.0, cov_func=None
    ),
    "adstock_alpha": Prior("Beta", alpha=2, beta=3, dims="channel"),
    "saturation_lam": Prior("Gamma", alpha=2, beta=2, dims="channel"),
    "saturation_beta": Prior("HalfNormal", sigma=cost_share, dims="channel"),
}

Observe that we are going to use a Gaussian process to model the non-linear trend component (see {ref}`mmm_tv_intercept`).

Let's fit the model!

Both models use 50 basis terms for the time-varying baseline, so the baseline approximation is held fixed in the comparison.

In [ ]:
%%time

mmm = MMM(
    adstock=GeometricAdstock(l_max=4),
    saturation=LogisticSaturation(),
    date_column="date",
    channel_columns=["x1", "x2"],
    target_column="y",
    time_varying_intercept=True,
    time_varying_media=False,
    yearly_seasonality=5,
    model_config=model_config,
)

y = model_df["y"]
X = model_df.drop(columns=["y"])

sampler_config = {
    "tune": 2_000,
    "chains": 4,
    "draws": 2_000,
    "nuts_sampler": "pymc",
    "cores": 4,
    "target_accept": 0.97,
    "random_seed": rng,
}

mmm.build_model(X, y)
mmm.add_original_scale_contribution_variable(
    var=[
        "channel_contribution",
        "fourier_contribution",
        "intercept_contribution",
    ]
)
_ = mmm.fit(X, y, **sampler_config)
_ = mmm.sample_posterior_predictive(
    X, extend_idata=True, combined=True, random_seed=rng
)

Before interpreting the posterior, inspect divergences, R-hat, and effective sample sizes for every free parameter.

In [ ]:
def convergence_summary(mmm):
    diagnostics = az.summary(
        mmm.idata,
        var_names=[var.name for var in mmm.model.free_RVs],
        kind="diagnostics",
        round_to="none",
    )
    numeric = diagnostics[["r_hat", "ess_bulk", "ess_tail"]].apply(
        pd.to_numeric, errors="coerce"
    )
    return pd.Series(
        {
            "divergences": int(mmm.idata["sample_stats"]["diverging"].sum().item()),
            "max_r_hat": numeric["r_hat"].max(),
            "min_ess_bulk": numeric["ess_bulk"].min(),
            "min_ess_tail": numeric["ess_tail"].min(),
        },
        dtype=float,
    )


convergence_summary(mmm)

Next, we look into the model components contributions:

In [ ]:
fig, axes = mmm.plot.contributions_over_time(
    var=[
        "channel_contribution_original_scale",
        "intercept_contribution_original_scale",
        "fourier_contribution_original_scale",
    ],
    dims={"channel": ["x1", "x2"]},
    combine_dims=True,
    hdi_prob=0.94,
    figsize=(12, 7),
)
sns.lineplot(x="date", y="y", data=model_df, color="black", label="y", ax=axes[0, 0])
legend = axes[0, 0].get_legend()
legend.set_bbox_to_anchor((0.8, -0.12))

The results look very similar to the results from the original blog post. In particular, note that we were able to capture the non-linear trend.

Next, we look into the channel contributions against the true effects (which we know from the data generating process).

In [ ]:
channels_contribution_original_scale = mmm.idata["posterior"][
    "channel_contribution_original_scale"
]
channels_contribution_original_scale_hdi = az.hdi(
    data=channels_contribution_original_scale
)
fig, ax = plt.subplots(
    nrows=2, figsize=(15, 8), ncols=1, sharex=True, sharey=False, layout="constrained"
)
amplitude = 100
for i, x in enumerate(["x1", "x2"]):
    # HDI estimated contribution in the original scale
    ax[i].fill_between(
        x=model_df["date"],
        y1=channels_contribution_original_scale_hdi.sel(channel=x, ci_bound="lower"),
        y2=channels_contribution_original_scale_hdi.sel(channel=x, ci_bound="upper"),
        color=f"C{i}",
        label=rf"{x} $94\%$ HDI contribution",
        alpha=0.4,
    )
    sns.lineplot(
        x="date",
        y=f"{x}_effect",
        data=raw_df.assign(**{f"{x}_effect": lambda df: amplitude * df[f"{x}_effect"]}),  # noqa B023
        color=f"C{i}",
        label=f"{x} effect",
        ax=ax[i],
    )
    ax[i].legend(loc="center left", bbox_to_anchor=(1, 0.5))
    ax[i].set(title=f"Channel {x}")

We see that the contribution for `x1` is very different from the true effect. This is because the absence of the unobserved confounder `z`. For `x2`, the contribution is very similar to the true effect.

Finally, we can compute the ROAS for the `x1` and `x2` channels (again, ignoring the small carryover effect).


In [ ]:
roas_posterior = mmm.incrementality.contribution_over_spend(
    frequency="all_time"
).rename("roas")
pc = azp.plot_dist(
    roas_posterior.to_dataset(name="roas"),
    col_wrap=1,
    figure_kwargs={
        "figsize": (12, 7),
        "sharex": True,
        "sharey": False,
        "layout": "constrained",
    },
)
fig = pc.viz["/"]["figure"].values.item()
axes = fig.axes
axes[0].axvline(
    true_roas_x1, color="black", linestyle="--", linewidth=3, label="True ROAS"
)
axes[0].legend(loc="upper right")
axes[0].set(title="Channel $x_{1}$")
axes[1].axvline(
    true_roas_x2, color="black", linestyle="--", linewidth=3, label="True ROAS"
)
axes[1].legend(loc="upper right")
axes[1].set(title="Channel $x_{2}$", xlabel="ROAS")
fig.suptitle("ROAS Posterior Distribution", fontsize=16, fontweight="bold");

We see that the ROAS for `x1` is very different from the true value. This is reflecting the bias induced by the unobserved confounder `z`. The models suggests that `x1` is more effective than `x2`, but we know from the data generating process that `x2` is more effective!

## Lift Test Model

Now we fit a model with some lift tests. We will use the same model configuration as before, but we free the priors of the beta channel coefficients as these are included in the saturation function parametrization. In general, we expect lift test priors or associated custom likelihoods to be better than the cost share prior.

In [ ]:
model_config = {
    "likelihood": Prior("Normal", sigma=Prior("HalfNormal", sigma=2)),
    "gamma_fourier": Prior("Normal", mu=0, sigma=2, dims="fourier_mode"),
    "intercept_tvp_config": HSGPKwargs(
        m=50, L=None, eta_lam=1.0, ls_mu=5.0, ls_sigma=10.0, cov_func=None
    ),
    "adstock_alpha": Prior("Beta", alpha=2, beta=3, dims="channel"),
    "saturation_lam": Prior("Gamma", alpha=2, beta=2, dims="channel"),
    "saturation_beta": Prior("HalfNormal", sigma=cost_share, dims="channel"),
}

mmm_lift = MMM(
    adstock=GeometricAdstock(l_max=4),
    saturation=LogisticSaturation(),
    date_column="date",
    channel_columns=["x1", "x2"],
    target_column="y",
    time_varying_intercept=True,
    time_varying_media=False,
    yearly_seasonality=5,
    model_config=model_config,
)

# we need to build the model before adding the lift test measurements
mmm_lift.build_model(X, y)

```{admonition} Lift Tests
:class: tip

In a lift study, one temporarily changes the budget of a channel for a fixed period of time, and then uses some method (for example [CausalPy](https://causalpy.readthedocs.io/en/stable/)) to make inference about the change in sales directly caused by the adjustment.

A lift test is characterized by:

- `channel`: the channel that was tested
- `x`: pre-test channel spend
- `delta_x`: change made to x
- `delta_y`: inferred change in sales due to delta_x
- `sigma`: standard deviation of delta_y

An experiment characterized in this way can be viewed as two points on the saturation curve for the channel. 
```

The four synthetic lift measurements cover two spend levels for each channel. Their dates will determine when each measurement becomes available in cross-validation.

In [ ]:
df_lift_test = pd.DataFrame(
    data={
        "channel": ["x1", "x2", "x1", "x2"],
        "x": [0.25, 0.1, 0.8, 0.25],
        "delta_x": [0.25, 0.1, 0.8, 0.25],
        "delta_y": [
            true_roas_x1 * 0.25,
            true_roas_x2 * 0.1,
            true_roas_x1 * 0.8,
            true_roas_x2 * 0.25,
        ],
        "sigma": [3, 3, 3, 3],
        "date": pd.to_datetime(
            [
                X["date"].max() - pd.Timedelta(weeks=50),
                X["date"].max() - pd.Timedelta(weeks=30),
                X["date"].max() - pd.Timedelta(weeks=14),
                X["date"].max() - pd.Timedelta(weeks=12),
            ]
        ),
    }
)

df_lift_test

```{admonition} Comparison with the original blog post
:class: tip

Note that we have added the true ROAS for the `x1` and `x2` channels implicit to the `df_lift_test` table. We add them by multiplying the `delta_y` as this is what we would have observed if we had run the lift test (or similar values).

In the simulation [Media Mix Model and Experimental Calibration: A Simulation Study](https://juanitorduz.github.io/mmm_roas/), the author included these "true" values into the prior for the ROAS.
```

We now fit the model with the default signed Normal likelihood for the lift estimates. This likelihood retains the sign of each observed lift rather than calibrating to its absolute value.

In [ ]:
mmm_lift.add_lift_test_measurements(df_lift_test=df_lift_test)

In [ ]:
mmm_lift.add_original_scale_contribution_variable(
    var=[
        "channel_contribution",
        "fourier_contribution",
        "intercept_contribution",
    ]
)
_ = mmm_lift.fit(X, y, **sampler_config)
_ = mmm_lift.sample_posterior_predictive(
    X, extend_idata=True, combined=True, random_seed=rng
)

Inspect the calibrated model using the same sampling diagnostics.

In [ ]:
convergence_summary(mmm_lift)

Let's plot the components contributions as we did before.

In [ ]:
fig, axes = mmm_lift.plot.contributions_over_time(
    var=[
        "channel_contribution_original_scale",
        "intercept_contribution_original_scale",
        "fourier_contribution_original_scale",
    ],
    dims={"channel": ["x1", "x2"]},
    combine_dims=True,
    hdi_prob=0.94,
    figsize=(12, 7),
)
sns.lineplot(x="date", y="y", data=model_df, color="black", label="y", ax=axes[0, 0])
legend = axes[0, 0].get_legend()
legend.set_bbox_to_anchor((0.8, -0.12))

As before, we have recovered the non-linear trend component and the yearly seasonality.

Now, let's compare the channel contributions to the true ones.

In [ ]:
channels_contribution_original_scale = mmm_lift.idata["posterior"][
    "channel_contribution_original_scale"
]
channels_contribution_original_scale_hdi = az.hdi(
    data=channels_contribution_original_scale, prob=0.94
)
fig, ax = plt.subplots(
    nrows=2, figsize=(15, 8), ncols=1, sharex=True, sharey=False, layout="constrained"
)
amplitude = 100
for i, x in enumerate(["x1", "x2"]):
    # HDI estimated contribution in the original scale
    ax[i].fill_between(
        x=model_df["date"],
        y1=channels_contribution_original_scale_hdi.sel(channel=x, ci_bound="lower"),
        y2=channels_contribution_original_scale_hdi.sel(channel=x, ci_bound="upper"),
        color=f"C{i}",
        label=rf"{x} $94\%$ HDI contribution",
        alpha=0.4,
    )
    sns.lineplot(
        x="date",
        y=f"{x}_effect",
        data=raw_df.assign(**{f"{x}_effect": lambda df: amplitude * df[f"{x}_effect"]}),  # noqa B023
        color=f"C{i}",
        label=f"{x} effect",
        ax=ax[i],
    )
    ax[i].legend(loc="center left", bbox_to_anchor=(1, 0.5))
    ax[i].set(title=f"Channel {x}")

Compare the refreshed contribution intervals with the synthetic channel effects. Lift calibration adds information about the spend-response relationship, but agreement with these effects still needs to be checked rather than assumed.

Finally, let's compute the ROAS for the `x1` and `x2` channels.


In [ ]:
roas_posterior = mmm_lift.incrementality.contribution_over_spend(
    frequency="all_time"
).rename("roas")
pc = azp.plot_dist(
    roas_posterior.to_dataset(name="roas"),
    col_wrap=1,
    figure_kwargs={
        "figsize": (12, 7),
        "sharex": True,
        "sharey": False,
        "layout": "constrained",
    },
)
fig = pc.viz["/"]["figure"].values.item()
axes = fig.axes
axes[0].axvline(
    true_roas_x1, color="black", linestyle="--", linewidth=3, label="True ROAS"
)
axes[0].legend(loc="upper right")
axes[0].set(title="Channel $x_{1}$")
axes[1].axvline(
    true_roas_x2, color="black", linestyle="--", linewidth=3, label="True ROAS"
)
axes[1].legend(loc="upper right")
axes[1].set(title="Channel $x_{2}$", xlabel="ROAS")
fig.suptitle("ROAS Posterior Distribution", fontsize=16, fontweight="bold");

The lift-calibrated posterior can be compared with the synthetic truth in the plot above. A change in channel ranking or a narrower interval does not by itself establish that the interval covers the true all-time ROAS.

## Time Slice Cross Validation

We compare the baseline and lift-calibrated models in five expanding training windows, each followed by a 12-week forecast. The forecast windows overlap, so their scores are not independent replications.

For the calibrated model, `df_lift_test` supplies the lift measurements. Each fold uses only rows dated on or before the final training date. In a real application this date must represent when the completed estimate was available for calibration; a test's start date alone is not sufficient.

In [ ]:
%%time

cv = TimeSliceCrossValidator(
    n_init=115,
    forecast_horizon=12,
    date_column="date",
    step_size=1,
)
cv.plot_suite = "new"


cv_results, mmms = cv.run(
    X=X,
    y=y,
    mmm=mmm,
    original_scale_vars=["channel_contribution", "y"],
    sampler_config=sampler_config | {"progressbar": False},
    return_models=True,
)

cv_results_lift, mmms_lift = cv.run(
    X=X,
    y=y,
    mmm=mmm_lift,
    df_lift_test=df_lift_test,
    lift_test_date_column="date",
    original_scale_vars=["channel_contribution", "y"],
    sampler_config=sampler_config | {"progressbar": False},
    return_models=True,
)

fold_diagnostics = pd.concat(
    {
        "baseline": pd.DataFrame([convergence_summary(model) for model in mmms]),
        "lift_calibrated": pd.DataFrame(
            [convergence_summary(model) for model in mmms_lift]
        ),
    },
    names=["model", "fold"],
)
fold_diagnostics

### Parameter Development over Folds

Let's start by looking into how the parameters develop over the folds for both models. We are interested in assesing if they change significantly and if so, understand the reason behind it.

In [ ]:
# Baseline model
fig, ax = cv.plot.param_stability(
    cv_data=cv_results, var_names=["saturation_beta"], figsize=(12, 5)
)
fig.suptitle(
    "Saturation Beta over Folds - Baseline Model",
    fontsize=18,
    fontweight="bold",
    y=1.03,
)

# Calibrated model
fig, ax = cv.plot.param_stability(
    cv_data=cv_results_lift, var_names=["saturation_beta"], figsize=(12, 5)
)
fig.suptitle(
    "Saturation Beta over Folds - Calibrated Model",
    fontsize=18,
    fontweight="bold",
    y=1.03,
);

Inspect the parameter trajectories alongside the eligible lift rows reported below. A shift between folds can reflect both newly available observations and calibration evidence; it should not be attributed to a new experiment unless eligibility changes in that fold.

Let's look into the experiment dates and the fold boundaries to make this more concrete.

In [ ]:
fig, ax = plt.subplots()
sns.lineplot(x="date", y="y", data=model_df, color="black", label="y", ax=ax)


for fold_idx in range(len(cv_results["cv_metadata"]["metadata"])):
    start_date_test_fold = (
        cv_results["cv_metadata"]["metadata"]
        .sel(cv=f"Iteration {fold_idx}")
        .values.item()["X_test"]["date"]
        .min()
    )
    ax.axvline(start_date_test_fold, color="gray", linestyle="--")

for row in df_lift_test.itertuples():
    color = "C0" if row.channel == "x1" else "C1"
    ax.axvline(row.date, color=color, linestyle="--")

ax.legend()
ax.set_title(
    "Lift Test Measurements and Fold Boundaries", fontsize=18, fontweight="bold"
);

The gray lines mark the start of each forecast window. A lift measurement is eligible only when its date is on or before the preceding training end. The table below reports the exact boundaries and eligible rows, so it is possible to check whether any new lift evidence enters during these five folds.

In [ ]:
eligibility_rows = []
for fold_idx, metadata in enumerate(cv_results["cv_metadata"]["metadata"].values):
    train_end = metadata["X_train"]["date"].max()
    eligible = df_lift_test.loc[df_lift_test["date"] <= train_end]
    eligibility_rows.append(
        {
            "fold": fold_idx,
            "train_end": train_end,
            "forecast_start": metadata["X_test"]["date"].min(),
            "forecast_end": metadata["X_test"]["date"].max(),
            "eligible_lift_rows": len(eligible),
            "eligible_lift_dates": ", ".join(eligible["date"].dt.strftime("%Y-%m-%d")),
        }
    )
pd.DataFrame(eligibility_rows).set_index("fold")

We can continue the analysis by looking into the lambda parameters, which have a direct interpretation as the saturation effect.

In [ ]:
# Baseline model
fig, ax = cv.plot.param_stability(
    cv_data=cv_results, var_names=["saturation_lam"], figsize=(12, 5)
)
fig.suptitle(
    "Saturation Lambda over Folds - Baseline Model",
    fontsize=18,
    fontweight="bold",
    y=1.03,
)

# Calibrated model
fig, ax = cv.plot.param_stability(
    cv_data=cv_results_lift, var_names=["saturation_lam"], figsize=(12, 5)
)
fig.suptitle(
    "Saturation Lambda over Folds - Calibrated Model",
    fontsize=18,
    fontweight="bold",
    y=1.03,
);

Compare the lambda trajectories between models. The amount of fold-to-fold movement describes sensitivity to the expanding training sample; the eligibility table identifies whether newly available lift evidence could also explain that movement.

Now we look into a metric of interest: the ROAS. The following code just computes the ROAS for each fold.

In [ ]:
def get_fold_roas(models: list[MMM], fold_idx: int) -> DataArray:
    return (
        models[fold_idx]
        .incrementality.contribution_over_spend(frequency="all_time")
        .rename("roas")
    )


fold_roas = [get_fold_roas(mmms, i) for i in range(len(mmms))]

fold_roas_lift = [get_fold_roas(mmms_lift, i) for i in range(len(mmms_lift))]

Let's visualize the results

In [ ]:
def plot_fold_forest(folds, channel, true_value, title):
    """Forest plot of per-fold ROAS estimates for a single channel."""
    model_names = [f"Iteration {i}" for i in range(len(folds))]
    combined = xr.concat([f.to_dataset() for f in folds], dim="model").assign_coords(
        model=model_names
    )
    dt = xr.DataTree.from_dict({"/posterior": xr.DataTree(combined)})
    pc = azp.plot_forest(
        dt.sel(channel=channel),
        combined=True,
        figure_kwargs={"figsize": (10, 5)},
    )
    ax = pc.viz["plot"].sel(column="forest").item()
    ax.axvline(true_value, color="black", linestyle="--")
    ax.set(xlabel="ROAS", xlim=(80, 500))

    color_map = pc.aes["color"]["mapping"]
    handles = [
        Line2D([0], [0], color=str(color_map.sel(model=m).values), linewidth=3, label=m)
        for m in model_names
    ]
    handles.append(Line2D([0], [0], color="black", linestyle="--", label="true ROAS"))
    ax.legend(handles=handles, loc="lower right", fontsize=12)

    fig = pc.viz["/"]["figure"].values.item()
    fig.suptitle(title, fontsize=16, fontweight="bold")
    return pc


plot_fold_forest(fold_roas, "x1", true_roas_x1, "Baseline MMM (x1)")
plot_fold_forest(fold_roas_lift, "x1", true_roas_x1, "Calibrated MMM (x1)")
plot_fold_forest(fold_roas, "x2", true_roas_x2, "Baseline MMM (x2)")
plot_fold_forest(fold_roas_lift, "x2", true_roas_x2, "Calibrated MMM (x2)");

Compare the baseline and calibrated intervals with the synthetic reference line for each channel. Experimental calibration and forecast accuracy answer different questions; neither interval stability nor low prediction error alone proves that a causal estimate is correct.

### Out-of-sample Predictions

Finally, we want to look into the out-of-sample predictions of the models for each fold. To quantify the performance of the models, we consider the Continuous Ranked Probability Score (CRPS). This metric is a generalization of the mean absolute error (MAE) to the probabilistic predictions setting. Intuitively, the CRPS measures the “distance” between the predicted distribution and the observed value by comparing their cumulative distribution function (CDF) at every point on the real line. For more details, take a look into the blog post ["Intuition behind CRPS"](https://juanitorduz.github.io/crps/).

Let's compute the CRPS for each fold and visualize the results.

In [ ]:
def get_fold_crps(cv_results: xr.DataTree, fold_idx: int) -> float:
    dates = (
        cv_results["cv_metadata"]["metadata"]
        .sel(cv=f"Iteration {fold_idx}")
        .values.item()["X_test"]["date"]
        .to_numpy()
    )
    y_true = (
        cv_results["cv_metadata"]["metadata"]
        .sel(cv=f"Iteration {fold_idx}")
        .values.item()["y_test"]
    )
    y_pred = (
        az.extract(cv_results, group="posterior_predictive")["y_original_scale"]
        .sel(cv=f"Iteration {fold_idx}")
        .transpose(..., "date")
        .sel(date=dates)
    )
    return crps(y_true, y_pred)


folds_crps = [
    get_fold_crps(cv_results, i)
    for i in range(len(cv_results["cv_metadata"]["metadata"]))
]
folds_crps_lift = [
    get_fold_crps(cv_results_lift, i)
    for i in range(len(cv_results_lift["cv_metadata"]["metadata"]))
]
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(
    np.arange(len(folds_crps)),
    folds_crps,
    marker="o",
    markersize=10,
    color="C0",
    label="Baseline MMM",
)
ax.plot(
    np.arange(len(folds_crps_lift)),
    folds_crps_lift,
    marker="o",
    markersize=10,
    color="C1",
    label="Calibrated MMM",
)
ax.legend()
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.set(xlabel="Fold", ylabel="CRPS")
ax.set_title("CRPS per Fold", fontsize=18, fontweight="bold");

In [ ]:
crps_by_fold = pd.DataFrame(
    {
        "baseline_crps": folds_crps,
        "calibrated_crps": folds_crps_lift,
    }
).rename_axis("fold")
crps_by_fold["calibrated_minus_baseline"] = (
    crps_by_fold["calibrated_crps"] - crps_by_fold["baseline_crps"]
)
crps_by_fold

The table and plot report prediction scores for the same five forecast windows; lower CRPS is better. Compare the refreshed values rather than assuming that calibration improves forecasts. Forecast performance evaluates the sales distribution, while experimental evidence informs the causal channel response.

## Conclusion

This synthetic example shows why good sales predictions can coexist with biased channel ROAS when a confounder is unobserved. Lift measurements add experimental evidence to the observational model through a signed Normal likelihood.

Interpret the refreshed posterior, contribution intervals, fold diagnostics, and CRPS together. Calibration does not guarantee recovery of synthetic truth or an improvement in forecasts. Cross-validation must respect when completed experiment estimates became available, and overlapping forecast windows should not be treated as independent experiments.

In [ ]:
%load_ext watermark
%watermark -n -u -v -iv -w -p pymc_marketing,pytensor